In [1]:
# inspecting the brain draft and make consenus
import os
import pandas as pd
import scipy
import cobra
from cobra.io import load_matlab_model
from cobra.io import load_model
from cobra.flux_analysis import gapfill
from cobra.io import write_sbml_model
from cobra.io import save_matlab_model

# inspect the tissue drafts built with four different algorithms before getting into the consensus reconstruction

In [2]:
# read the drafts model and the universal model
corda = load_matlab_model('/Users/eso1993/Library/CloudStorage/Box-Box/PFOS_Project/CORDA_mouse_models/CORDA_iMiceBrain.mat')
ftinit = load_matlab_model('/Users/eso1993/Library/CloudStorage/Box-Box/PFOS_Project/ftINIT_mouse_models/ftINIT_iMiceBrain.mat')
mCADRE = load_matlab_model('/Users/eso1993/Library/CloudStorage/Box-Box/PFOS_Project/mCADRE_output_brain_draft_reconstruction/iMiceBrain_mCADRE.mat')
fastacormics = load_matlab_model('/Users/eso1993/Library/CloudStorage/Box-Box/PFOS_Project/rFASTCOROMICS_mouse_models/FASTCORMICS_iMiceBrain.mat')
universal = load_matlab_model('/Users/eso1993/Library/CloudStorage/Box-Box/PFOS_Project/mouse_model_iMM1865/iMM1865_updated.mat')

Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x
No defined compartments in model modelBrain. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x


In [3]:
# inspect the metabolites associated with the objective function in the universal model
objective_rxn = universal.reactions.get_by_id("BIOMASS_reaction")
for met in objective_rxn.metabolites:
    print(met.id)

h2o[c]
atp[c]
adp[c]
h[c]
pi[c]
glu__L[c]
asp__L[c]
gtp[c]
asn__L[c]
ala__L[c]
cys__L[c]
gln__L[c]
gly[c]
ser__L[c]
thr__L[c]
lys__L[c]
arg__L[c]
met__L[c]
pail_hs[c]
ctp[c]
pchol_hs[c]
pe_hs[c]
chsterol[c]
pglyc_hs[c]
clpn_hs[c]
dgtp[n]
dctp[n]
datp[n]
utp[c]
dttp[n]
g6p[c]
his__L[c]
tyr__L[c]
ile__L[c]
leu__L[c]
trp__L[c]
phe__L[c]
pro__L[c]
ps_hs[c]
sphmyln_hs[c]
val__L[c]


In [4]:
# modify the metabolite format in corda 
import re
for met in corda.metabolites:
    new_id = re.sub(r'_(\w+)$', r'[\1]', met.id)
    met.id = new_id

In [5]:
# inspect if these metabolites are existed in the draft_model
missing_mets = []

for met in objective_rxn.metabolites:
    if met.id not in corda.metabolites:
        missing_mets.append(met.id)

print("Missing metabolite IDs:", missing_mets) 

# all metabolites are absent in corda model 
# all metabolites are present in ftinit, mCADRE, and fastacormics

Missing metabolite IDs: ['glu__L[c]', 'asp__L[c]', 'asn__L[c]', 'ala__L[c]', 'cys__L[c]', 'gln__L[c]', 'ser__L[c]', 'thr__L[c]', 'lys__L[c]', 'arg__L[c]', 'met__L[c]', 'pail_hs[c]', 'pchol_hs[c]', 'pe_hs[c]', 'pglyc_hs[c]', 'clpn_hs[c]', 'his__L[c]', 'tyr__L[c]', 'ile__L[c]', 'leu__L[c]', 'trp__L[c]', 'phe__L[c]', 'pro__L[c]', 'ps_hs[c]', 'sphmyln_hs[c]', 'val__L[c]']


In [6]:
# ONLY for CORDA model
#  add these metabolites and associated reactions
# 1. Add missing metabolites to the model
for met in objective_rxn.metabolites:
    if met.id not in corda.metabolites:
        corda.add_metabolites([met.copy()])  # add clean copies

# 2. Find and add universal reactions connected to those metabolites
target_met_ids = [met.id for met in objective_rxn.metabolites]
reactions_to_add = []

for rxn in universal.reactions:
    rxn_met_ids = [met.id for met in rxn.metabolites]
    if any(met_id in rxn_met_ids for met_id in target_met_ids):
        if rxn.id not in corda.reactions:
            reactions_to_add.append(rxn.copy())

corda.add_reactions(reactions_to_add)

print(f"Added {len(reactions_to_add)} reactions.")

# 870 reactions were added for the corda draft model to allow having all the objective function metabolites 


Added 870 reactions.


In [7]:
# inspect the statistics of the brain-draft models
print("corda model:")
print("  Reactions:", len(corda.reactions))
print("  Genes:", len(corda.genes))
print("  Metabolites:", len(corda.metabolites))

print("ftinit model:")
print("  Reactions:", len(ftinit.reactions))
print("  Genes:", len(ftinit.genes))
print("  Metabolites:", len(ftinit.metabolites))

print("mCADRE model:")
print("  Reactions:", len(mCADRE.reactions))
print("  Genes:", len(mCADRE.genes))
print("  Metabolites:", len(mCADRE.metabolites))

print("fastacormics model:")
print("  Reactions:", len(fastacormics.reactions))
print("  Genes:", len(fastacormics.genes))
print("  Metabolites:", len(fastacormics.metabolites))

corda model:
  Reactions: 8277
  Genes: 1839
  Metabolites: 5209
ftinit model:
  Reactions: 8539
  Genes: 1727
  Metabolites: 5401
mCADRE model:
  Reactions: 5066
  Genes: 1427
  Metabolites: 3637
fastacormics model:
  Reactions: 5106
  Genes: 1564
  Metabolites: 3604


In [ ]:
# inspect blocked reactions in each draft
blocked_corda = cobra.flux_analysis.find_blocked_reactions(corda)
print("corda model:")
print(len(blocked_corda)) # 1161 blocked

blocked_mCADRE = cobra.flux_analysis.find_blocked_reactions(mCADRE)
print("mCADRE model:")
print(len(blocked_mCADRE)) # zero blocked

blocked_fastacormics = cobra.flux_analysis.find_blocked_reactions(fastacormics)
print("fastacormics model:")
print(len(blocked_fastacormics)) # zero blocked 

blocked_ftinit = cobra.flux_analysis.find_blocked_reactions(ftinit)
print("ftinit model:")
print(len(blocked_ftinit)) # 621 blocked reactions


Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp07maavqf.lp
Reading time = 0.02 seconds
: 5210 rows, 16555 columns, 68465 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp9snfuint.lp
Reading time = 0.02 seconds
: 5210 rows, 16555 columns, 68465 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmplzmh3p8u.lp
Reading time = 0.03 seconds
: 5210 rows, 16555 columns, 68465 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use 

In [9]:
# inspect if the objective function reaction is among the blocked ones in draft model
if "BIOMASS_reaction" in blocked_corda:
    print("yap it is here")
else:
    print("nop")

# objective function is not among the blocked reactions in either ftinit or corda models so we can remov these blocked reactions safely

nop


In [10]:
# remove the blocked reactions from CORDA model and then inspect the biomass_Reactions mets 
corda.remove_reactions(blocked_corda)
missing_mets = []

for met in objective_rxn.metabolites:
    if met.id not in corda.metabolites:
        missing_mets.append(met.id)

print("Missing metabolite IDs:", missing_mets) # all metabolites are present 

/opt/anaconda3/envs/python396_env/lib/python3.9/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


Missing metabolite IDs: []


In [11]:
# remove the blocked reactionsfrom ftinit model and then inspect the biomass_Reactions mets 
ftinit.remove_reactions(blocked_ftinit)
missing_mets_ftinit = []

for met in objective_rxn.metabolites:
    if met.id not in ftinit.metabolites:
        missing_mets_ftinit.append(met.id)

print("Missing metabolite IDs:", missing_mets_ftinit) # all metabolites are present 

Missing metabolite IDs: []


In [12]:
# inspect CORDA model again
print("corda model:")
print(len(corda.reactions), len(corda.metabolites), len(corda.genes))

# inspect ftinit model again
print("ftinit model:")
print(len(ftinit.reactions), len(ftinit.metabolites), len(ftinit.genes))


corda model:
7389 5209 1839
ftinit model:
7918 5401 1727


In [13]:
# inspect the solution of each draft model 
from cobra.flux_analysis import pfba
sol_corda = pfba(corda)
sol_ftinit = pfba(ftinit)
sol_mCADRE = pfba(mCADRE)
sol_fastcormics = pfba(fastacormics)

print("Corda:")
print(sol_corda.fluxes['BIOMASS_reaction'])
print("ftinit:")
print(sol_ftinit.fluxes['BIOMASS_reaction'])
print("mCADRE:")
print(sol_mCADRE.fluxes['BIOMASS_reaction'])
print("fastcormics:")
print(sol_fastcormics.fluxes['BIOMASS_reaction'])



Corda:
626.5072155005336
ftinit:
306.2126447072629
mCADRE:
34.72014362487381
fastcormics:
219.93739075645036


In [26]:
# export refined drafts of corda and ftinit 
save_matlab_model(corda, "/Users/egabal/Desktop/corda_iMiceBrain_prerefined.mat")
write_sbml_model(corda, "/Users/egabal/Desktop/corda_iMiceBrain_prerefined.xml")

save_matlab_model(ftinit, "/Users/egabal/Desktop/ftinit_iMiceBrain_prerefined.mat")
write_sbml_model(ftinit, "/Users/egabal/Desktop/ftinit_iMiceBrain_prerefined.xml")

## now we define the shared genes, rxns, and metabolites among the four drafts and build a consensus model of that

In [14]:
# Get shared gene IDs
# Get shared gene IDs
shared_genes = list(
    set(g.id for g in corda.genes) &
    set(g.id for g in mCADRE.genes) &
    set(g.id for g in ftinit.genes) &
    set(g.id for g in fastacormics.genes)
)
print("Number of shared genes:", len(shared_genes))

# Get shared reaction IDs 
shared_rxns = list(
    set(r.id for r in corda.reactions) &
    set(r.id for r in mCADRE.reactions) &
    set(r.id for r in ftinit.reactions) &
    set(r.id for r in fastacormics.reactions)
)
print("Number of shared reactions:", len(shared_rxns))

# Get shared metabolite IDs 
shared_mets = list(
    set(m.id for m in corda.metabolites) &
    set(m.id for m in mCADRE.metabolites) &
    set(m.id for m in ftinit.metabolites) &
    set(m.id for m in fastacormics.metabolites)
)
print("Number of shared metabolites:", len(shared_mets))


Number of shared genes: 1307
Number of shared reactions: 2865
Number of shared metabolites: 2185


In [15]:
from cobra import Model

# Build an empty model
consensus_model = Model("model")

# Add only the shared reactions from the universal model
for rxn_id in shared_rxns:
    if rxn_id in universal.reactions:
        consensus_model.add_reactions([universal.reactions.get_by_id(rxn_id).copy()])

# Filtering Step : Keep reactions with shared metabolites 
shared_met_ids = set(shared_mets)

reactions_met_filtered = [
    rxn for rxn in consensus_model.reactions
    if any(met.id in shared_met_ids for met in rxn.metabolites)
]

# Replace model with metabolite-filtered reactions
consensus_model.remove_reactions([rxn for rxn in consensus_model.reactions if rxn not in reactions_met_filtered])

# Final consensus model summary
print("Consensus Model (Filtered by Reactions + Metabolites):")
print("Reactions:", len(consensus_model.reactions))
print("Genes:", len(consensus_model.genes))
print("Metabolites:", len(consensus_model.metabolites))

Consensus Model (Filtered by Reactions + Metabolites):
Reactions: 2720
Genes: 1257
Metabolites: 2515


In [16]:
# again inspect the blocked reactions and number 
blocked = cobra.flux_analysis.find_blocked_reactions(consensus_model)
len(blocked)

Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp4p6597xb.lp
Reading time = 0.01 seconds
: 2516 rows, 5441 columns, 24227 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp1p4ws0dk.lp
Reading time = 0.01 seconds
: 2516 rows, 5441 columns, 24227 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpksqt11p7.lp
Reading time = 0.01 seconds
: 2516 rows, 5441 columns, 24227 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use onl

2429

In [17]:
# inspect the presence of objective function
if "BIOMASS_reaction" in consensus_model.reactions:
    print("YAP")
else:
    print("NOP")

YAP


In [18]:
if "BIOMASS_reaction" in blocked:
    print("Biomass is blocked")
else:
    print("NO")

Biomass is blocked


In [19]:
# inspect model solution
sol_brain = pfba(consensus_model)

print("consensus_model:")
print(sol_brain.fluxes['BIOMASS_reaction'])


consensus_model:
0.0


In [20]:
# identify the objective function associated metabolites 
missing_mets_brain = []

for met in objective_rxn.metabolites:
    if met.id not in consensus_model.metabolites:
        missing_mets_brain.append(met.id)

print("Missing metabolite IDs:", missing_mets_brain) 

Missing metabolite IDs: []


In [21]:
# check the blocked metabolites if they are produced or consumed 
blocked_mets = set()
for rxn_id in blocked:
    rxn = consensus_model.reactions.get_by_id(rxn_id)
    blocked_mets.update([met.id for met in rxn.metabolites])

# For each metabolite, check producing and consuming reactions
for met_id in blocked_mets:
    met = consensus_model.metabolites.get_by_id(met_id)
    producing = [rxn.id for rxn in met.reactions if met in rxn.products]
    consuming = [rxn.id for rxn in met.reactions if met in rxn.reactants]
    
    print(f"Metabolite: {met_id}")
    print(f"  Produced by: {producing if producing else 'None'}")
    print(f"  Consumed by: {consuming if consuming else 'None'}\n")

Metabolite: M00004[r]
  Produced by: ['HMR_2955']
  Consumed by: ['HMR_3636']

Metabolite: gd2_hs[c]
  Produced by: ['HMR_0851']
  Consumed by: ['HMR_0843']

Metabolite: pmtcoa[x]
  Produced by: ['FA160COAabcp_1', 'ACACT8p']
  Consumed by: None

Metabolite: ibupcoa__R[c]
  Produced by: ['IBUP_RASCL1hep']
  Consumed by: ['IBUP_Rshep']

Metabolite: ksi_deg30[l]
  Produced by: ['GALASE11ly']
  Consumed by: None

Metabolite: hepdeceth[e]
  Produced by: ['HEPDECETH']
  Consumed by: ['EX_hepdeceth_e']

Metabolite: hxcoa[x]
  Produced by: None
  Consumed by: ['HMR_3098', 'C6COAt']

Metabolite: CE5151[r]
  Produced by: ['HMR_2967']
  Consumed by: ['HMR_3648']

Metabolite: M01165[m]
  Produced by: ['HMR_6908']
  Consumed by: ['HMR_6909']

Metabolite: CE4841[c]
  Produced by: ['RE3161C']
  Consumed by: ['RE3162C']

Metabolite: ksii_core2_pre5[g]
  Produced by: ['AG13T2g']
  Consumed by: ['S6T1g']

Metabolite: CE0693[m]
  Produced by: ['RE1527M']
  Consumed by: ['RE1534M']

Metabolite: octdececrn

In [22]:
# add the reactions associated with these blocked metabolites 
associated_rxns = set()
for met_id in blocked_mets:
    if met_id in universal.metabolites:
        met = universal.metabolites.get_by_id(met_id)
        for rxn in met.reactions:
            associated_rxns.add(rxn.id)

# Add those reactions to the consensus model if not already present
for rxn_id in associated_rxns:
    if rxn_id in universal.reactions and rxn_id not in consensus_model.reactions:
        consensus_model.add_reactions([universal.reactions.get_by_id(rxn_id).copy()])

print("Updated Consensus Model:")
print("Reactions:", len(consensus_model.reactions))
print("Genes:", len(consensus_model.genes))
print("Metabolites:", len(consensus_model.metabolites))

Updated Consensus Model:
Reactions: 8280
Genes: 1855
Metabolites: 5378


In [23]:
# again inspect the blocked reactions and number 
blocked = cobra.flux_analysis.find_blocked_reactions(consensus_model)
len(blocked)

Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpdj0d1nfo.lp
Reading time = 0.02 seconds
: 5379 rows, 16561 columns, 74241 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmps1yk90c1.lp
Reading time = 0.03 seconds
: 5379 rows, 16561 columns, 74241 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp_i40mqzg.lp
Reading time = 0.03 seconds
: 5379 rows, 16561 columns, 74241 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use 

2261

In [24]:
if "BIOMASS_reaction" in blocked:
    print("Biomass is blocked")
else:
    print("NO")

NO


In [25]:
# Biomass_Reaction is not in the blocked reactiosn so remove them
consensus_model.remove_reactions(blocked)
print("Reactions:", len(consensus_model.reactions))
print("Genes:", len(consensus_model.genes))
print("Metabolites:", len(consensus_model.metabolites))

Reactions: 6019
Genes: 1855
Metabolites: 5378


In [28]:
if "BIOMASS_reaction" in consensus_model.reactions:
    print("YEAH it is here")
else:
    print("NOP")

YEAH it is here


In [26]:
# try running gap filling for the biomass_Reaction
consensus_model.solver = 'gurobi'
with consensus_model:
    consensus_model.objective = consensus_model.reactions.get_by_id('BIOMASS_reaction')
    solution = gapfill(consensus_model, universal, iterations=1, exchange_reactions=True, demand_reactions=True)
print(solution)

Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpzafufp9g.lp
Reading time = 0.02 seconds
: 5378 rows, 12038 columns, 52548 nonzeros
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpw68zqg2o.lp
Reading time = 0.03 seconds
: 5839 rows, 21224 columns, 80812 nonzeros
[[]]


In [27]:
# Check for Blocked Biomass Precursors
biomass_rxn = consensus_model.reactions.get_by_id("BIOMASS_reaction")

print("Checking biomass precursors...\n")
for met in biomass_rxn.reactants:
    met_obj = consensus_model.metabolites.get_by_id(met.id)
    producers = [rxn.id for rxn in met_obj.reactions if met_obj in rxn.products]
    
    if not producers:
        print(f"❌ {met.id} has no producing reactions.")
    else:
        print(f"✅ {met.id} is produced by: {producers}")

Checking biomass precursors...

✅ ps_hs[c] is produced by: ['PSt3', 'PSFLIP', 'PSSA2_hs', 'PSSA1_hs']
✅ val__L[c] is produced by: ['r1863', 'r1855', 'r1854', 'r1865', 'r1659', 'VALt4', 'r1853', 'r1976', 'r1852', 'r1658', 'r1862', 'VALB0AT3tc', 'r1856', 'r1861', 'r1972', 'r1973', 'r1858', 'r1860', 'VALtec', 'VALLAT1tc', 'r1974', 'r1864', 'r1859', 'r1975', 'r1657', 'VALATB0tc', 'r1857', 'VALPHELAT2tc', 'r1866']
✅ h2o[c] is produced by: ['HMR_1681', 'DESAT18_10', 'RE0567C', 'HMR_2289', 'HMR_2336', 'HMR_6874', 'RE3122C', 'PPPGO', 'RE1233C', 'APOCF', 'PPBNGS', 'DESAT18_8', 'DESAT18', 'HMR_2218', 'HMR_1944', 'FAS140COA', 'FAS100COA', 'RE3570C', 'FAS80COA_L', 'DESAT16_2', 'DOPABMO_1', 'ALOX52', 'r0511', 'HMR_1981', 'H2OGLYAQPt', 'QUILSYN', 'HMR_1735', 'HMR_2581', 'HMR_7197', 'r0210', 'R01115', 'RE3251C', 'CYSTS', 'r0145', 'HMR_2344', 'HMR_0981', 'DESAT18_9', 'HMR_1737', 'HMGCOAS', 'RE3235C', 'RE3434C', 'MELATNOX', 'HMR_2472', 'HMR_2284', 'NOS1', 'MDRPD', 'FPGS_1', 'FAS180', 'r0647', 'RE1925C'

In [28]:
# inspect certain reactions of interest 
reactions_interest = ["ATPS4mi", "GLUt6"]

for rxn_id in reactions_interest:
    if rxn_id in consensus_model.reactions:
        print(f"{rxn_id}: ✅ Present in model")
    else:
        print(f"{rxn_id}: ❌ Not found in model")

ATPS4mi: ✅ Present in model
GLUt6: ✅ Present in model


In [32]:
# gapfill for the GLUT6
consensus_model.solver = 'gurobi'
with consensus_model:
    consensus_model.objective = consensus_model.reactions.get_by_id('GLUt6')
    solution = gapfill(consensus_model, universal, iterations=1, exchange_reactions=True, demand_reactions=True)
print(solution)

Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp2_gkumbt.lp
Reading time = 0.02 seconds
: 5378 rows, 12038 columns, 52548 nonzeros
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp9jjk56x1.lp
Reading time = 0.03 seconds
: 5839 rows, 21224 columns, 80812 nonzeros
[[]]


In [32]:
consensus_model.add_reactions([universal.reactions.get_by_id("ATPS4mi")])
consensus_model.solver = 'gurobi'
with consensus_model:
    consensus_model.objective = consensus_model.reactions.get_by_id('ATPS4mi')
    solution = gapfill(consensus_model, universal, iterations=1, exchange_reactions=True, demand_reactions=True)
print(solution)

Ignoring reaction 'ATPS4mi' since it already exists.


Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpfukhtbgh.lp
Reading time = 0.02 seconds
: 5337 rows, 10222 columns, 44938 nonzeros
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp7247oynp.lp
Reading time = 0.03 seconds
: 5839 rows, 21224 columns, 80812 nonzeros
[[]]


In [29]:
#  run FBA individually to check the functionality 
from cobra.flux_analysis import flux_variability_analysis

for rxn_id in ["ATPS4mi", "GLUt6", "BIOMASS_reaction"]:
    rxn = consensus_model.reactions.get_by_id(rxn_id)
    with consensus_model:
        consensus_model.objective = rxn
        fba_solution = consensus_model.optimize()
        print(f"{rxn_id}: Flux = {fba_solution.objective_value}")

ATPS4mi: Flux = 1000.0
GLUt6: Flux = 1000.0
BIOMASS_reaction: Flux = 422.0943775817407


In [30]:
# assing these two reactions in the objecive function as well
consensus_model.objective=["BIOMASS_reaction", "ATPS4mi", "GLUt6"]
print(consensus_model.objective)

Maximize
1.0*ATPS4mi - 1.0*ATPS4mi_reverse_d277b + 1.0*BIOMASS_reaction - 1.0*BIOMASS_reaction_reverse_5a818 + 1.0*GLUt6 - 1.0*GLUt6_reverse_10912


In [31]:
#Setting upper bound at 80% so biomass_reaction can carry flux, before 422.09437758174073
consensus_model.reactions.get_by_id("GLUt6").upper_bound = 337.67
consensus_model.reactions.get_by_id("ATPS4mi").upper_bound = 337.67

# inspect the optimal solution at each reaction
from cobra.flux_analysis import pfba
sol = pfba(consensus_model)
print(sol.fluxes['ATPS4mi'])
print(sol.fluxes['GLUt6'])
print(sol.fluxes['BIOMASS_reaction'])

337.67
337.67
402.9848999210031


In [32]:
# export the models after refining and then test through memote
from cobra.io import write_sbml_model
from cobra.io import save_matlab_model
save_matlab_model(consensus_model, "/Users/eso1993/Desktop/iMiceBrain_consensus.mat")
write_sbml_model(consensus_model, "/Users/eso1993/Desktop/iMiceBrain_consensus.xml")
